# 13 — The redundancy reveal

Notebook `12` ended on a win and a question. The quantum measures rose with the injected coupling $K$ exactly as the classical baselines did — *it looked like it works*. But "rises with $K$" is the bar `11` set, and PLV already clears it. So here we ask the sharper question head-on: **does the quantum measure read anything PLV does not?** The answer we find is a real, clean result — and it points us straight at the fix.

**Prerequisites:** `12_naive_phase_embedding_tracks_K` (the naive phase qubit, $\rho_{AB}$, and the four-measure sweep).

**What you'll be able to do**
- Turn "does it add anything?" into a measurable test: **rank-correlate** each quantum measure against PLV across the $K$-sweep with `scipy.stats.spearmanr`.
- Find that $\mathrm{Spearman}(\mathrm{PLV}, \mathrm{QMI}) \approx 1$ and $\mathrm{Spearman}(\mathrm{PLV}, \mathrm{Bures}) \approx 1$ — the naive quantum measures are **rank-identical to PLV**, i.e. PLV in disguise.
- Explain *why* from the operator itself: on this embedding the only coupling-dependent part of $\rho_{AB}$ is the off-block coherence, whose magnitude **is** the PLV, while the marginals stay pinned at $I/2$ — so QMI and Bures are deterministic **functions of PLV**.
- Read this not as a failure of quantum OT but as a precise diagnosis of the *naive embedding* — the exact problem statement the next notebooks set out to solve.

This is the turning point of the capstone. By the end you will know, sharply and for a structural reason, why the naive quantum measures cannot beat PLV — and that knowledge is what makes the fix that follows feel inevitable rather than lucky.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.composite import partial_trace
from qot_course.quantum_ot.capstone import (
    simulate_kuramoto_dyad,
    joint_density_matrix,
    plv,
    sweep_coupling_measures,
)

np.random.seed(42)
viz.use_course_style()

## 1. The question, made sharp

In `12` every measure — quantum and classical — rose with the injected coupling $K$. That is genuinely necessary: a coupling statistic that *ignored* $K$ would be useless. But it is not sufficient, and here is the reason. The whole motivation for carrying the dyad across the bridge into a quantum state was the hope that a quantum-information or quantum-transport measure would see *more* than a phase-only baseline — that it would resolve coupling structure PLV is blind to. "Rises with $K$" cannot demonstrate that, because **PLV rises with $K$ too**. Two measures can both track the ground truth and still be the same measurement in different clothing.

So we sharpen the question into something testable:

> Across the sweep, does the quantum measure carry information PLV lacks — or does it merely **re-encode PLV**?

The clean test is a **rank correlation**. If QMI is a strictly increasing function of PLV across the sweep — whenever PLV goes up, QMI goes up, with no exceptions — then QMI orders the dyads exactly as PLV does. It draws no distinction PLV did not already draw. Spearman's $\rho$ measures precisely this: it is the Pearson correlation of the *ranks*, so it equals $1$ when one variable is any monotone-increasing transform of the other, regardless of the curve's shape (Spearman, 1904). A Spearman of $\approx 1$ between PLV and a quantum measure is therefore the signature of redundancy: same ranking, no new information.

We run this on the **same transition-focused grid** as `12` — $K \in [0, 0.5]$ with 16 points — so that PLV spans a wide range and the test has something to bite on.

## 2. The measurement: rank-correlate the quantum measures against PLV

`sweep_coupling_measures` runs the full pipeline for each $K$ on the grid — simulate the dyad, embed the phases into $\rho_{AB}$, evaluate all four measures — and hands back the four curves. We feed PLV and each quantum measure to `spearmanr` and read off the rank correlation. Then we *draw* the relationship: a scatter of each quantum measure against PLV across the sweep. If the points fall on a single rising curve, the quantum measure is a function of PLV.

In [ ]:
# Same transition-focused grid as notebook 12: the dyad locks by K ~ 0.5, so this
# window is where PLV (and every other measure) actually moves.
k_grid = np.linspace(0.0, 0.5, 16)
sweep = sweep_coupling_measures(k_grid, duration=200.0, seed=0)

print("grid:", k_grid.shape[0], "couplings from", f"{k_grid[0]:.2f}", "to", f"{k_grid[-1]:.2f}")
print(f"PLV spans [{sweep['plv'].min():.3f}, {sweep['plv'].max():.3f}]")
print()

rho_qmi, _ = spearmanr(sweep["plv"], sweep["qmi"])
rho_bures, _ = spearmanr(sweep["plv"], sweep["bures"])
print(f"Spearman(PLV, QMI)   = {rho_qmi:.3f}")
print(f"Spearman(PLV, Bures) = {rho_bures:.3f}")

**Read the output.** The grid puts PLV across a broad span — from barely above its noise floor near $K = 0$ to near-saturation by $K = 0.5$ — so there is plenty of room for a quantum measure to disagree with it about the *ordering* of the dyads. It does not. Both rank correlations come back at $\approx 1.00$: as PLV rises across the sweep, QMI rises with it without a single inversion, and so does Bures. To three decimals the quantum measures rank the sixteen dyads in **exactly** the order PLV does. Whatever distinctions PLV can draw, these quantum measures draw the same ones — and no others. Let us see it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))

# QMI vs PLV
axes[0].plot(
    sweep["plv"], sweep["qmi"], "o", color=COLORS["quantum"],
    markersize=7, markeredgecolor="white", markeredgewidth=0.8, zorder=3,
)
axes[0].set_xlabel("phase-locking value  PLV")
axes[0].set_ylabel("quantum mutual information  (nats)")
axes[0].set_title("QMI vs PLV across the sweep", pad=8)

# Bures vs PLV
axes[1].plot(
    sweep["plv"], sweep["bures"], "o", color=COLORS["source"],
    markersize=7, markeredgecolor="white", markeredgewidth=0.8, zorder=3,
)
axes[1].set_xlabel("phase-locking value  PLV")
axes[1].set_ylabel("Bures-coupling")
axes[1].set_title("Bures vs PLV across the sweep", pad=8)

fig.suptitle(
    rf"The naive quantum measures are rank-identical to PLV "
    rf"(Spearman $\approx$ {rho_qmi:.2f})",
    fontsize=14, fontweight="bold", y=1.02,
)
fig.tight_layout()
plt.show()

**Read the figure.** Two scatter plots, one per quantum measure, each placing PLV on the horizontal axis and the quantum measure on the vertical, with one point per coupling $K$ on the sweep.

In **both** panels the sixteen points lie along a single, tight, monotonically rising curve — not a cloud. As PLV grows from left to right, QMI (left, sky blue) climbs without ever dipping; Bures (right, periwinkle) does the same. There is no scatter *around* the curve and no fold-back: fix PLV and you have fixed the quantum measure. That is the visual meaning of a Spearman of $\approx 1$ — the quantum measure is a **function of PLV** on this embedding. The curve is not a straight line (QMI bends; Spearman does not care about shape, only order), but it is single-valued and increasing, which is all redundancy needs. Read plainly: on the naive embedding, the quantum measure had no independent access to the coupling. It was handed PLV and re-expressed it.

## 3. Why — the coupling lives entirely in one coherence, and that coherence is PLV

A Spearman of $1$ on one grid could be a coincidence of this particular run. It is not — and the reason is structural, readable straight off the operator we built in `12`. This is the heart of the reveal, so let us go slowly.

Write the naive joint state at one instant. With $|\psi(t)\rangle = (|0\rangle + e^{i\theta(t)}|1\rangle)/\sqrt 2$, the product $|\psi_A\rangle \otimes |\psi_B\rangle$ has amplitude vector

$$ \tfrac{1}{2}\bigl(\,1,\; e^{i\theta_B},\; e^{i\theta_A},\; e^{i(\theta_A+\theta_B)}\,\bigr) \quad\text{in the basis } |00\rangle,|01\rangle,|10\rangle,|11\rangle. $$

Time-averaging the projector gives $\rho_{AB} = \mathbb{E}_t[\,|\Psi\rangle\langle\Psi|\,]$. Look at where the coupling can possibly enter. The **diagonal** is $\mathbb{E}_t[\tfrac14(1,1,1,1)] = \tfrac14 I_4$ — independent of $K$, as we saw in `12`. The **single-qubit marginals** depend on $\mathbb{E}_t[e^{i\theta_A}]$ and $\mathbb{E}_t[e^{i\theta_B}]$; because each oscillator's own phase drifts around the full circle, these average to $\approx 0$, so $\rho_A \approx \rho_B \approx I/2$ regardless of $K$. The one matrix element that *moves* with coupling is the off-block coherence linking $|01\rangle$ and $|10\rangle$:

$$ \rho_{AB}[1,2] \;=\; \mathbb{E}_t\!\left[\tfrac14\, e^{i\theta_B} e^{-i\theta_A}\right] \;=\; \tfrac14\,\overline{\langle e^{i(\theta_A-\theta_B)}\rangle}, \qquad\text{so}\qquad \bigl|\rho_{AB}[1,2]\bigr| \;=\; \tfrac14\,\mathrm{PLV}. $$

The off-block coherence magnitude **is** the PLV, up to the fixed factor $\tfrac14$. So across the sweep, $\rho_{AB}$ has essentially **one** coupling-dependent degree of freedom, and that degree of freedom is PLV. Any function of $\rho_{AB}$ — QMI, Bures, anything — is therefore a function of PLV alone. Not approximately: by the structure of the embedding. Let us confirm the identity numerically.

In [ ]:
print(f"{'K':>6}  {'PLV':>7}  {'|rho[1,2]|':>11}  {'PLV / 4':>9}  {'|rho_A off-diag|':>16}")
print("-" * 58)
for K in k_grid[::3]:
    t1, t2 = simulate_kuramoto_dyad(K=float(K), duration=200.0, seed=0)
    rho = joint_density_matrix(t1, t2)
    rho_a = partial_trace(rho, keep=[0], dims=[2, 2])
    plv_K = plv(t1, t2)
    coherence = abs(rho[1, 2])
    marginal_offdiag = abs(rho_a[0, 1])
    print(f"{K:>6.3f}  {plv_K:>7.3f}  {coherence:>11.4f}  {plv_K / 4:>9.4f}  {marginal_offdiag:>16.4f}")

**Read the output.** Three things to see, column by column.

First, compare the **`|rho[1,2]|`** column against **`PLV / 4`**: they match to every printed digit, at every coupling. The off-block coherence is not merely correlated with PLV — it *equals* PLV divided by four, the exact identity the algebra above predicts. The quantum object's coupling content is PLV, relabelled.

Second, the **`|rho_A off-diag|`** column stays small — a few hundredths — across the whole sweep, never tracking $K$. The single-qubit marginal sits at $\approx I/2$ throughout, so it contributes no coupling-dependent information. The marginals are inert; only the off-block coherence carries the signal.

Third, put those together. $\rho_{AB}$'s diagonal is frozen at $\tfrac14 I_4$, its marginals are frozen at $I/2$, and its one moving part is the coherence $= \mathrm{PLV}/4$. A density matrix whose only coupling-dependent freedom is PLV can feed a coupling measure nothing *but* PLV. That is why QMI and Bures came back rank-identical to it — they are deterministic functions $f(\mathrm{PLV})$, and the scatter in §2 is the graph of $f$. The quantum measure never had independent access to the coupling; the embedding fed it PLV and only PLV.

## 4. The diagnosis is a signpost, not a wall

It is worth naming exactly what we have found, and exactly what we have *not*.

We have **not** found that quantum optimal transport fails to read coupling, nor that QMI and Bures are bad measures. They computed correctly; they tracked the ground truth; they agree with the classical workhorse to three decimals. The earlier version of this analysis stopped at this point and filed it under an *open question* — "does the quantum machinery offer a real advantage? we do not know" — as though the redundancy were a dead end. It is not, and reading it that way mistakes a property of the *embedding* for a property of *quantum OT*.

Here is the precise statement. The redundancy is a fact about the **naive phase qubit**, not about quantum coupling measures in general. We chose the simplest possible map — one phase to one equatorial qubit — and that map is so thin that the resulting $\rho_{AB}$ has a single coupling-dependent number in it, which happens to be PLV. Feed a quantum measure a state that only knows PLV, and of course it can only report PLV. The bottleneck is upstream of the measure, in how we turned the signal into a state.

That diagnosis is constructive, because it tells us *exactly* what to change. If we want $\rho_{AB}$ to carry structure beyond PLV — so that a quantum measure has something genuinely new to read — we must enrich the embedding so the coupling lands in *more* of the operator than one coherence whose magnitude is pinned to the phase-locking value. That is the precise problem statement the next notebooks take up: `14` enriches the embedding so $\rho_{AB}$ encodes more than PLV, and `15` puts the enriched measure to the discriminating test that the naive one could never pass. The reveal is not where the capstone runs out of road — it is where it learns where the road is.

## Your turn

**Warm-up.** Re-run the §2 measurement with a different `seed` (say `seed=1` or `seed=7`) in `sweep_coupling_measures`, keeping the same grid. Predict before you run: will the two Spearman values still be $\approx 1$? Explain your prediction using §3 — does changing the noise realisation change the *fact* that the off-block coherence equals $\mathrm{PLV}/4$, or only the particular numbers along the curve?

**Go further.** Argue, without running any new code, that **any** real-valued function of $\rho_{AB}$ must be rank-correlated $1$ with PLV on this embedding. Start from the §3 facts — the diagonal is $\tfrac14 I_4$, the marginals are $\approx I/2$, the only moving element is $\rho_{AB}[1,2]$ with $|\rho_{AB}[1,2]| = \mathrm{PLV}/4$ — and reason about how many coupling-dependent degrees of freedom a measure could possibly depend on. What would have to be *false* about the embedding for a quantum measure to break the tie with PLV?

**Challenge.** The fix in `14` will enrich the embedding so that $\rho_{AB}$ carries coupling structure in places other than the single $|01\rangle\langle10|$ coherence. Predict, in words, what would have to show up in the §2 scatter for the enrichment to count as a success: describe the shape of a QMI-vs-PLV scatter for which $\mathrm{Spearman}(\mathrm{PLV}, \mathrm{QMI})$ drops *below* $1$, and say what such a departure from the curve would mean about two dyads that share a PLV but differ in higher-order structure.

## What you built

- You turned a vague worry — "does the quantum measure add anything?" — into a **sharp, falsifiable test**: rank-correlate each quantum measure against PLV across the sweep, where a Spearman of $1$ means *same ordering, no new information*.
- You ran it and got a clean result: $\mathrm{Spearman}(\mathrm{PLV}, \mathrm{QMI}) \approx 1$ and $\mathrm{Spearman}(\mathrm{PLV}, \mathrm{Bures}) \approx 1$. The scatter plots confirmed it visually — the points fall on a single rising curve, so the naive quantum measures are **PLV in disguise**.
- You proved *why* from the operator itself: on the naive embedding the diagonal is frozen at $\tfrac14 I_4$, the marginals at $I/2$, and the lone coupling-dependent element is the off-block coherence with $|\rho_{AB}[1,2]| = \mathrm{PLV}/4$ — so QMI and Bures are deterministic **functions of PLV**, by construction rather than by coincidence.
- You reframed that redundancy correctly: it is a diagnosis of the *naive embedding*, not a verdict on quantum OT — a precise statement of what must change for a quantum measure to see beyond PLV.

This is a real result, and a sharp one: you now know, for a structural reason you can write down in one line, why the naive quantum measures cannot beat PLV. That is exactly the foothold the fix needs. In `14_richer_embedding_applied` we enrich the embedding so $\rho_{AB}$ encodes coupling structure beyond PLV — giving the quantum measure, for the first time, something the phase-locking value cannot already see.

## References

- C. Spearman, "The proof and measurement of association between two things", *American Journal of Psychology* **15**, 72–101 (1904). DOI:10.2307/1412159 — the rank correlation used here; equals $1$ for any monotone-increasing relationship.
- J.-P. Lachaux, E. Rodriguez, J. Martinerie & F. J. Varela, "Measuring phase synchrony in brain signals", *Human Brain Mapping* **8**, 194–208 (1999). DOI:10.1002/(SICI)1097-0193(1999)8:4<194::AID-HBM4>3.0.CO;2-C — the phase-locking value, $\mathrm{PLV} = |\langle e^{i(\theta_A-\theta_B)}\rangle|$, here shown to equal $4\,|\rho_{AB}[1,2]|$.
- D. Trevisan, *Optimal transport methods for quantum systems* (arXiv:2202.02091, 2022). DOI:10.48550/arXiv.2202.02091 — quantum-transport coupling measures of the kind tested here.

**Previous:** `notebooks/04_QuantumOptimalTransport/12_naive_phase_embedding_tracks_K.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/14_richer_embedding_applied.ipynb`